In [22]:
import pandas as pd 
import seaborn as sns 
import numpy as np 
import matplotlib.pyplot as plt
import warnings 
warnings.filterwarnings('ignore')

In [23]:
df=pd.read_csv('train.txt',sep=';',header=None,names=['Text','emotions'])

In [24]:
df.head()

,Text,emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [25]:
df.isnull().sum()

Text        0
emotions    0
dtype: int64

In [26]:
df.duplicated().sum()

np.int64(1)

In [27]:
df.shape

(16000, 2)

In [28]:
df=df.drop_duplicates()

In [29]:
df.duplicated().sum()

np.int64(0)

In [30]:
df.shape

(15999, 2)

In [31]:
df['emotions'].value_counts()

emotions
joy         5361
sadness     4666
anger       2159
fear        1937
love        1304
surprise     572
Name: count, dtype: int64

In [32]:
df['emotions'].unique()

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [33]:
emo_map={
    "sadness":0,
    "anger":1,
    "love":2,
    "surprise":3,
    "fear":4,
    "joy":5
}
df['emotions']=df['emotions'].map(emo_map)

In [34]:
df['emotions'].value_counts()

emotions
5    5361
0    4666
1    2159
4    1937
2    1304
3     572
Name: count, dtype: int64

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15999 entries, 0 to 15999
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Text      15999 non-null  object
 1   emotions  15999 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 375.0+ KB


In [36]:
print(df["Text"].str.islower().all())

True


In [37]:
df['Text']=df['Text'].str.lower()

In [38]:
import string 
def rem_punc(txt): 
    return txt.translate(str.maketrans('','',string.punctuation))
    

In [39]:
df['Text']=df['Text'].apply(rem_punc)

In [40]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['Text'] = df['Text'].apply(remove_emojis)

In [43]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [44]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\arpan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arpan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [45]:
stop_words = set(stopwords.words('english'))

In [46]:
print(stop_words)

{'over', 'most', "we'd", 'an', 'what', "haven't", 's', 'above', "you'd", 'these', 'each', 'as', 'on', 'in', 'from', 'yours', 'being', 'not', "they'll", "weren't", "he's", 'at', 'doing', 'do', 'doesn', "didn't", 'hadn', 'than', "aren't", 'some', 'won', 'he', 'yourselves', 'that', 'any', "she'll", 'just', 'for', "he'll", 'o', "it's", 'up', 'they', "they're", "hadn't", "mightn't", 'myself', 'don', 'didn', 'll', 'off', 'before', 'to', 'needn', 'will', 'other', 'having', 'be', 'during', 'hasn', 'of', "they've", 'too', 've', 'which', 'this', 'you', 'was', "they'd", "don't", "wouldn't", 'mightn', 'ourselves', 'whom', 'more', 'himself', 'here', "he'd", "should've", "we'll", "i've", "she's", 'if', 'down', 'ours', 'where', 'but', 'now', 'we', 'hers', "isn't", 'me', "that'll", 'a', "won't", 'ma', 'your', 'once', 'there', "it'd", 'so', 'those', 'shouldn', "you'll", "needn't", 'with', 'isn', 'his', "mustn't", 'into', 'them', 'few', "shan't", 'been', 'how', 'below', 'nor', 'her', "i'm", "you're", "w

In [47]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\arpan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [48]:
df['Text'].loc[565]

'i have been feeling conflicted on whether or not i as a follower of christ should celebrate the ever popular pagan originated modern day holidays'

In [49]:
from nltk.tokenize import word_tokenize

def remove_stopwords(text):
    words = word_tokenize(text)

    cleaned = [
        word for word in words
        if word not in stop_words
    ]

    return " ".join(cleaned)

In [50]:
df["Text"] = df["Text"].apply(remove_stopwords)

In [51]:
df['Text'].loc[565]

'feeling conflicted whether follower christ celebrate ever popular pagan originated modern day holidays'

In [52]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [53]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['Text'], df['emotions'], test_size=0.20, random_state=42)

In [54]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)


nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)


pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))

0.773125


In [55]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0], shape=(3200,))

In [56]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [57]:
y_pred = nb2_model.predict(X_test_tfidf)

In [58]:
print(accuracy_score(y_test, y_pred))

0.6634375


In [59]:
from sklearn.linear_model import LogisticRegression

In [60]:
logistic_model = LogisticRegression(max_iter=1000)

In [61]:
logistic_model.fit(X_train_tfidf,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [62]:
log_pred = logistic_model.predict(X_test_tfidf)

In [63]:
print(accuracy_score(y_test,log_pred ))

0.86125


In [64]:
from sklearn.metrics import classification_report

print(classification_report(y_test, log_pred))

              precision    recall  f1-score   support

           0       0.91      0.94      0.92       950
           1       0.91      0.81      0.86       439
           2       0.92      0.60      0.73       303
           3       0.89      0.47      0.62       106
           4       0.87      0.76      0.81       375
           5       0.80      0.96      0.87      1027

    accuracy                           0.86      3200
   macro avg       0.88      0.76      0.80      3200
weighted avg       0.87      0.86      0.86      3200



In [65]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, log_pred))

[[895  14   2   0   5  34]
 [ 29 354   0   0  13  43]
 [  9   4 182   0   3 105]
 [ 12   0   1  50  19  24]
 [ 26  13   1   5 284  46]
 [ 17   3  12   1   3 991]]


In [74]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    C=0.5
)

In [75]:
logistic_model.fit(
    X_train_tfidf,
    y_train
)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.5
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [76]:
log_pred = logistic_model.predict(
    X_test_tfidf
)

In [77]:
from sklearn.metrics import accuracy_score

print(
    accuracy_score(
        y_test,
        log_pred
    )
)

0.8821875


In [78]:

from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        log_pred
    )
)

              precision    recall  f1-score   support

           0       0.95      0.89      0.92       950
           1       0.86      0.90      0.88       439
           2       0.76      0.93      0.84       303
           3       0.61      0.92      0.73       106
           4       0.87      0.81      0.84       375
           5       0.92      0.88      0.90      1027

    accuracy                           0.88      3200
   macro avg       0.83      0.89      0.85      3200
weighted avg       0.89      0.88      0.88      3200



In [79]:
from sklearn.metrics import confusion_matrix

print(
    confusion_matrix(
        y_test,
        log_pred
    )
)

[[844  35  18   7  16  30]
 [  9 396   1   3  14  16]
 [  0   2 281   1   3  16]
 [  3   0   1  97   5   0]
 [  9  11   3  37 303  12]
 [ 23  14  66  13   9 902]]


In [80]:
train_pred = logistic_model.predict(
X_train_tfidf
)

from sklearn.metrics import accuracy_score

print(
accuracy_score(
y_train,
train_pred
)
)

0.9261661067270881


In [81]:
print(
    logistic_model.score(
        X_train_tfidf,
        y_train
    )
)

print(
    logistic_model.score(
        X_test_tfidf,
        y_test
    )
)

0.9261661067270881
0.8821875


In [82]:
import pickle

pickle.dump(
    logistic_model,
    open("sentiment_model.pkl", "wb")
)

In [84]:
pickle.dump(
    tfidf_vectorizer,
    open("tfidf.pkl", "wb")
)

In [85]:
model = pickle.load(
    open("sentiment_model.pkl","rb")
)

vectorizer = pickle.load(
    open("tfidf.pkl","rb")
)

print("Loaded Successfully")

Loaded Successfully
